# Tarea 24 · Verificación anti-leakage

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/24_verificacion_antileakage.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel.

Auditamos las tareas 21–23: separación de clientes, alineación de etiquetas y ajuste de transformaciones exclusivamente en entrenamiento. Contrastamos las matrices exportadas con el código de los notebooks 22 y 23, sin sobrescribir datos.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt` y ejecutar todas las celdas dentro del repositorio, después de completar las tareas 21–23.
- **Colab:** subir el repositorio descomprimido, conservando `README.md`, `notebooks/trabajo/` y `data/processed/`, y ubicarse en su raíz. Se requieren el dataset limpio, los cuatro CSV de `particion/`, los dos de `codificacion/` y los cuatro CSV finales.
- Este notebook lee los archivos locales; no descarga datos ni modifica las exportaciones.


## 1. Reconstrucción de la partición

Repetimos la división estratificada 80/20 con `random_state=42` sobre el dataset limpio y contrastamos todos los valores e índices con los CSV originales de la tarea 21. Así comprobamos también que ni el objetivo ni el identificador se incluyan como predictores.

In [1]:
from pathlib import Path
import ast
import copy
import io
import json
from contextlib import redirect_stdout
from unittest.mock import patch
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

actual = Path.cwd().resolve()
RAIZ = next((p for p in (actual, *actual.parents) if (p / 'README.md').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Ubicate dentro del repositorio, con README.md disponible.')
base = RAIZ / 'data' / 'processed'
def leer(carpeta, nombre):
    return pd.read_csv(carpeta / f'{nombre}.csv', index_col='fila_origen')

df = pd.read_csv(base / 'telco_clean.csv')
X = df.drop(columns='Churn')
y = df['Churn'].map({'No': 0, 'Yes': 1})
assert not y.isna().any()
assert not {'Churn', 'customerID', 'fila_origen'} & set(X.columns)
reconstruidos = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
originales = {}
for nombre, esperado in zip(['X_train', 'X_test', 'y_train', 'y_test'], reconstruidos):
    datos = leer(base / 'particion', nombre)
    esperado = esperado.to_frame() if isinstance(esperado, pd.Series) else esperado
    pd.testing.assert_frame_equal(datos.rename_axis(None), esperado)
    originales[nombre] = datos
train = originales['X_train']
test = originales['X_test']
assert train.index.is_unique and test.index.is_unique
assert train.index.intersection(test.index).empty
assert len(train) + len(test) == len(df)
assert abs(originales['y_train']['Churn'].mean() - originales['y_test']['Churn'].mean()) < 0.001
print(f'Partición reproducida: {len(train)} filas train y {len(test)} test; sin solapamiento.')

Partición reproducida: 5634 filas train y 1409 test; sin solapamiento.


La coincidencia con la partición reproducida confirma valores, etiquetas y orden de filas. La estratificación utiliza el objetivo para distribuir clases, como está previsto en el diseño; no estima parámetros de los predictores con test.

## 2. Auditoría de las llamadas de ajuste

Leemos el código real de los notebooks 22 y 23. Ejecutamos sus celdas en espacios separados, omitiendo únicamente las celdas de exportación que contienen `to_csv`. Interceptamos `ColumnTransformer` para registrar y validar cada `fit_transform` y `transform`: el primero debe recibir exactamente train y el segundo exactamente test. También exigimos que las llamadas explícitas a `fit` o `fit_transform` del código tengan como único argumento `X_train`. No se ajustan transformaciones con etiquetas.

El control depende de la estructura actual de esos notebooks: si se agregan otras formas de ajuste o se mezclan ajuste y exportación en una celda, habrá que ampliar la auditoría. No es una prueba universal para cualquier código futuro.

In [2]:
eventos = []
class ColumnTransformerAuditado(ColumnTransformer):
    def fit_transform(self, X, y=None, **params):
        pd.testing.assert_frame_equal(X, train)
        assert y is None, 'La codificación no debe aprender del objetivo.'
        eventos.append((tarea_actual, 'fit_transform', len(X)))
        return super().fit_transform(X, y=y, **params)

    def transform(self, X, **params):
        pd.testing.assert_frame_equal(X, test)
        antes = copy.deepcopy(self.named_transformers_)
        resultado = super().transform(X, **params)
        # Test debe aplicar los parámetros existentes, sin modificarlos.
        for nombre, bloque in antes.items():
            if isinstance(bloque, str):
                continue
            for atributo in ['mean_', 'var_', 'scale_', 'data_min_', 'data_max_', 'n_samples_seen_', 'categories_']:
                if hasattr(bloque, atributo):
                    valor = getattr(bloque, atributo)
                    nuevo = getattr(self.named_transformers_[nombre], atributo)
                    if atributo == 'categories_':
                        for a, b in zip(valor, nuevo):
                            np.testing.assert_array_equal(a, b)
                    else:
                        np.testing.assert_array_equal(valor, nuevo)
        eventos.append((tarea_actual, 'transform', len(X)))
        return resultado

contextos = {}
for tarea_actual, archivo in [(22, '22_codificacion.ipynb'), (23, '23_escalado.ipynb')]:
    notebook = json.loads((RAIZ / 'notebooks' / 'trabajo' / archivo).read_text(encoding='utf-8'))
    contexto = {'display': lambda *args, **kwargs: None}
    ajustes = 0
    with patch('sklearn.compose.ColumnTransformer', ColumnTransformerAuditado), redirect_stdout(io.StringIO()):
        for celda in notebook['cells']:
            if celda['cell_type'] != 'code':
                continue
            fuente = ''.join(celda['source'])
            arbol = ast.parse(fuente)
            llamadas = [n for n in ast.walk(arbol) if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute)]
            fits = [n for n in llamadas if n.func.attr in {'fit', 'fit_transform', 'partial_fit'}]
            exporta = any(n.func.attr == 'to_csv' for n in llamadas)
            assert not (exporta and fits), 'Separar ajuste y exportación antes de auditar.'
            if exporta:
                continue
            for llamada in fits:
                assert llamada.func.attr == 'fit_transform'
                assert len(llamada.args) == 1 and isinstance(llamada.args[0], ast.Name)
                assert llamada.args[0].id == 'X_train' and not llamada.keywords
                ajustes += 1
            # Suprimimos solo la visualización, sin alterar el cálculo.
            arbol.body = [n for n in arbol.body if not (isinstance(n, ast.ImportFrom) and n.module == 'IPython.display')]
            exec(compile(arbol, archivo, 'exec'), contexto)
    assert ajustes == (1 if tarea_actual == 22 else 2)
    contextos[tarea_actual] = contexto
assert eventos == [(22, 'fit_transform', len(train)), (22, 'transform', len(test)),
                   (23, 'fit_transform', len(train)), (23, 'fit_transform', len(train)),
                   (23, 'transform', len(test))]
display(pd.DataFrame(eventos, columns=['tarea', 'operación', 'filas']))

,tarea,operación,filas
0,22,fit_transform,5634
1,22,transform,1409
2,23,fit_transform,5634
3,23,fit_transform,5634
4,23,transform,1409


Las cinco operaciones registradas corresponden a la codificación de la tarea 22, los dos escaladores comparados con train en la tarea 23 y la transformación final de test. La selección de StandardScaler está fundamentada sin comparar resultados de modelos sobre test.

## 3. Parámetros aprendidos y archivos exportados

Contrastamos las categorías y estadísticas con entrenamiento. Verificamos los extremos del MinMaxScaler y la media y varianza del StandardScaler. Comparamos las matrices recalculadas con los archivos de las tareas 22 y 23, y las etiquetas finales con las originales. La igualdad de archivos complementa la inspección de llamadas; por sí sola no demostraría ausencia de fuga.

In [3]:
c22, c23 = contextos[22], contextos[23]
for contexto, clave in [(c22, 'codificador'), (c23, 'preprocessor'), (c23, 'preprocessor_minmax')]:
    bloque = contexto[clave]
    for columna, categorias in zip(contexto['columnas_nominales'], bloque.named_transformers_['nominal'].categories_):
        assert set(categorias) == set(train[columna].unique())
    assert bloque.named_transformers_['contrato'].categories_[0].tolist() == ['Month-to-month', 'One year', 'Two year']
numericas = c23['columnas_numericas']
standard = c23['preprocessor'].named_transformers_['numericas']
minmax = c23['preprocessor_minmax'].named_transformers_['numericas']
np.testing.assert_allclose(standard.mean_, train[numericas].mean())
np.testing.assert_allclose(standard.var_, train[numericas].var(ddof=0))
np.testing.assert_allclose(minmax.data_min_, train[numericas].min())
np.testing.assert_allclose(minmax.data_max_, train[numericas].max())
assert standard.n_samples_seen_ == minmax.n_samples_seen_ == len(train)
for contexto, carpeta, sufijo in [(c22, base / 'codificacion', 'codificado'), (c23, base, 'transformado')]:
    for nombre in ['X_train', 'X_test']:
        esperado = contexto[f'{nombre}_{sufijo}']
        recuperado = leer(carpeta, nombre)
        pd.testing.assert_frame_equal(recuperado, esperado, check_dtype=False, rtol=1e-12, atol=1e-12)
        assert np.isfinite(recuperado.to_numpy()).all()
        assert recuperado.index.equals(originales[nombre.replace('X_', 'y_')].index)
for nombre in ['y_train', 'y_test']:
    pd.testing.assert_frame_equal(leer(base, nombre), originales[nombre])
print('Parámetros de train y exportaciones de las tareas 22–23 verificados.')

Parámetros de train y exportaciones de las tareas 22–23 verificados.


## 4. Alcance de la revisión y orden de las operaciones

1. Limpieza determinista: conversión de tipos, imputación fija de `TotalCharges=0` cuando falta y `tenure=0`, eliminación del identificador y columnas calculadas por fila.
2. Separación de `Churn` y partición estratificada de clientes (tarea 21).
3. Ajuste de categorías, codificación y escaladores solo con train (tareas 22–23).
4. Transformación de test con parámetros congelados y exportación de matrices y etiquetas alineadas.
5. Entrenamiento futuro solo con train; evaluación final con test.

**Límites:** el EDA y el diagnóstico IQR se hicieron sobre el dataset completo antes de la partición. El criterio IQR no detectó valores atípicos y se conservaron todos los registros; esos umbrales no se usaron para recortar ni transformar los predictores. Las reglas fijas de limpieza y los cálculos por fila no estiman medias o extremos con test. Si se incorporan imputaciones estadísticas, selección de variables o tratamiento de outliers aprendido, deben ajustarse dentro de train.

El EDA global ya expuso información de la muestra de test: esta auditoría verifica el ajuste del preprocesamiento, pero no convierte ese conjunto en una muestra externa nunca observada. Evitar elegir variables, umbrales o modelos por sus resultados en test; para ajustes posteriores usar validación dentro de entrenamiento. La disponibilidad temporal de los cargos y la antigüedad al momento de predecir churn deberá definirse antes de usar el modelo en producción: estos controles no validan ese escenario.

## 5. Lista de comprobaciones

Las marcas describen comprobaciones realizadas por las celdas anteriores. Si alguna aserción falla, detener la ejecución y corregir el flujo antes de considerar aprobada esta lista.

- [x] Partición 80/20 reproducida con estratificación y `random_state=42`.
- [x] Índices únicos, sin clientes compartidos entre train y test.
- [x] `Churn`, `customerID` y `fila_origen` ausentes de los predictores.
- [x] Etiquetas alineadas y sin modificaciones en las exportaciones finales.
- [x] Las llamadas explícitas de ajuste revisadas reciben únicamente `X_train`.
- [x] One-Hot aprende categorías de train; el orden contractual es fijo.
- [x] StandardScaler y MinMaxScaler aprenden estadísticas de train.
- [x] Test solo se transforma y no modifica los parámetros ajustados.
- [x] Matrices exportadas coinciden con el preprocesamiento reproducido.
- [x] Alcance del EDA global y restricciones para el modelado documentados.

**Resultado:** el flujo actual de las tareas 21–23 respeta la separación train/test en el ajuste del preprocesamiento.